In [1]:
DATA_DIR = "/kaggle/input/datasets/iftekharuddin27/capstone-datasets" 

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
 
print("Imports loaded!")

Imports loaded!


In [3]:
df_en = pd.read_csv(f"{DATA_DIR}/English_combined_dataset.csv")
df_bn = pd.read_csv(f"{DATA_DIR}/bangla_hate_pool.csv")
 
print(f"English: {df_en.shape}")
print(f"Bangla: {df_bn.shape}")

English: (104737, 2)
Bangla: (83992, 3)


In [4]:
def preprocess_english(text):
    """
    Clean English text for NLP.
    NOTE: We do NOT remove stop words (important for sarcasm detection).
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # 1. Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    
    # 2. Remove @mentions (but keep the text after @)
    text = re.sub(r'@\w+', '', text)
    
    # 3. Remove RT (retweet marker)
    text = re.sub(r'\bRT\b', '', text)
    
    # 4. Remove HTML entities
    text = re.sub(r'&amp;', 'and', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&\w+;', '', text)
    
    # 5. Remove hashtag symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 6. Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?\'\"\-:;]', ' ', text)
    
    # 7. Convert to lowercase
    text = text.lower()
    
    # 8. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
 
# Apply preprocessing
print("Preprocessing English texts...")
df_en['text_clean'] = df_en['text'].apply(preprocess_english)
 
# Show before/after examples
print("\n" + "=" * 70)
print("ENGLISH PREPROCESSING - Before vs After")
print("=" * 70)
np.random.seed(42)
samples = df_en.sample(5)
for _, row in samples.iterrows():
    print(f"\n  BEFORE: {row['text'][:150]}")
    print(f"  AFTER:  {row['text_clean'][:150]}")
 
# Check for empty texts after cleaning
empty_count = (df_en['text_clean'].str.len() == 0).sum()
print(f"\nEmpty texts after cleaning: {empty_count}")
if empty_count > 0:
    # Remove empty texts
    df_en = df_en[df_en['text_clean'].str.len() > 0].reset_index(drop=True)
    print(f"Removed {empty_count} empty texts. New size: {df_en.shape}")

Preprocessing English texts...

ENGLISH PREPROCESSING - Before vs After

  BEFORE: write enough puppet code and you have the basics for ruby just to give the sit
  AFTER:  write enough puppet code and you have the basics for ruby just to give the sit

  BEFORE: sens. mccain, graham: trump's order could become 'self-inflicted wound' in terror fight
  AFTER:  sens. mccain, graham: trump's order could become 'self-inflicted wound' in terror fight

  BEFORE: new york to host 1998 ill-will games
  AFTER:  new york to host 1998 ill-will games

  BEFORE: area mom convinced 30-year-old daughter would be married by now if she just brushed her hair more
  AFTER:  area mom convinced 30-year-old daughter would be married by now if she just brushed her hair more

  BEFORE: @StephenStone4 you bad little bitch
  AFTER:  you bad little bitch

Empty texts after cleaning: 3
Removed 3 empty texts. New size: (104734, 3)


In [6]:
def preprocess_bangla(text):
    """
    Clean Bangla text for NLP.
    NOTE: We do NOT remove stop words.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # 1. Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    
    # 2. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    
    # 3. Remove hashtag symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 4. Remove HTML entities
    text = re.sub(r'&\w+;', '', text)
    
    # 5. Unicode normalization (NFC form - composed characters)
    text = unicodedata.normalize('NFC', text)
    
    # 6. Remove emojis and special Unicode symbols
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(' ', text)
    
    # 7. Keep Bangla characters, English letters, numbers, spaces, and basic punctuation
    # Bangla Unicode range: \u0980-\u09FF
    text = re.sub(r'[^\u0980-\u09FF\w\s.,!?\'\"\-:;।]', ' ', text)
    
    # 8. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
 
# Apply preprocessing
print("Preprocessing Bangla texts...")
df_bn['text_clean'] = df_bn['text'].apply(preprocess_bangla)
 
# Show before/after examples
print("\n" + "=" * 70)
print("BANGLA PREPROCESSING - Before vs After")
print("=" * 70)
np.random.seed(42)
samples = df_bn.sample(5)
for _, row in samples.iterrows():
    print(f"\n  BEFORE: {row['text'][:200]}")
    print(f"  AFTER:  {row['text_clean'][:200]}")
 
# Check for empty texts
empty_count = (df_bn['text_clean'].str.len() == 0).sum()
print(f"\nEmpty texts after cleaning: {empty_count}")
if empty_count > 0:
    df_bn = df_bn[df_bn['text_clean'].str.len() > 0].reset_index(drop=True)
    print(f"Removed {empty_count} empty texts. New size: {df_bn.shape}")

Preprocessing Bangla texts...

BANGLA PREPROCESSING - Before vs After

  BEFORE: এটা পাপন এর চাল
  AFTER:  এটা পাপন এর চাল

  BEFORE: এই খানকির বাচ্চারা তোরা আমার আল্লাহর কোরআনের যে আজকে কষ্ট জাহান্নামে জ্বলবে আগুন জ্বলবে
  AFTER:  এই খানকির বাচ্চারা তোরা আমার আল্লাহর কোরআনের যে আজকে কষ্ট জাহান্নামে জ্বলবে আগুন জ্বলবে

  BEFORE: দালালী চুদানোর জায়গা পাইলি না
  AFTER:  দালালী চুদানোর জায়গা পাইলি না

  BEFORE: মাশরাফি কিংবা সাকিব দুই জনই বর্তমান বাংলাদেশ দলের খেলোয়াড় ।
  AFTER:  মাশরাফি কিংবা সাকিব দুই জনই বর্তমান বাংলাদেশ দলের খেলোয়াড় ।

  BEFORE: আমি দৃষ্টান্ত মূলক শাস্ত্রী দাবী করছি ভারতের প্রশাসনের কাছে বাংলাদেশি মালয়েশিয়া থেকে
  AFTER:  আমি দৃষ্টান্ত মূলক শাস্ত্রী দাবী করছি ভারতের প্রশাসনের কাছে বাংলাদেশি মালয়েশিয়া থেকে

Empty texts after cleaning: 3
Removed 3 empty texts. New size: (83989, 4)


In [7]:
print("=" * 70)
print("DUPLICATE REMOVAL")
print("=" * 70)
 
en_before = len(df_en)
df_en = df_en.drop_duplicates(subset='text_clean', keep='first').reset_index(drop=True)
en_removed = en_before - len(df_en)
print(f"English: Removed {en_removed:,} duplicates. {en_before:,} -> {len(df_en):,}")
 
bn_before = len(df_bn)
df_bn = df_bn.drop_duplicates(subset='text_clean', keep='first').reset_index(drop=True)
bn_removed = bn_before - len(df_bn)
print(f"Bangla: Removed {bn_removed:,} duplicates. {bn_before:,} -> {len(df_bn):,}")
 
# Updated class distributions
print("\nUpdated English class distribution:")
print(df_en['class'].value_counts().sort_index())
print("\nUpdated Bangla label distribution:")
print(df_bn['label'].value_counts().sort_index())

DUPLICATE REMOVAL
English: Removed 415 duplicates. 104,734 -> 104,319
Bangla: Removed 227 duplicates. 83,989 -> 83,762

Updated English class distribution:
class
0    34698
1    34854
2    34767
Name: count, dtype: int64

Updated Bangla label distribution:
label
0    41399
1    25597
2    16766
Name: count, dtype: int64


In [8]:
print("=" * 70)
print("STRATIFIED SPLITTING (80% train / 10% val / 10% test)")
print("=" * 70)
 
def stratified_split(df, label_col, random_state=42):
    """
    Split dataframe into 80% train, 10% val, 10% test with stratification.
    """
    # First split: 80% train, 20% temp
    train_df, temp_df = train_test_split(
        df, test_size=0.2, random_state=random_state,
        stratify=df[label_col]
    )
    
    # Second split: 50% of temp = 10% val, 10% test
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=random_state,
        stratify=temp_df[label_col]
    )
    
    return train_df, val_df, test_df
 
# Split English
en_train, en_val, en_test = stratified_split(df_en, 'class')
print(f"\nENGLISH SPLIT:")
print(f"  Train: {len(en_train):,} ({len(en_train)/len(df_en)*100:.1f}%)")
print(f"  Val:   {len(en_val):,} ({len(en_val)/len(df_en)*100:.1f}%)")
print(f"  Test:  {len(en_test):,} ({len(en_test)/len(df_en)*100:.1f}%)")
print(f"\n  Train class distribution:")
print(f"  {en_train['class'].value_counts().sort_index().to_dict()}")
print(f"  Val class distribution:")
print(f"  {en_val['class'].value_counts().sort_index().to_dict()}")
print(f"  Test class distribution:")
print(f"  {en_test['class'].value_counts().sort_index().to_dict()}")
 
# Split Bangla
bn_train, bn_val, bn_test = stratified_split(df_bn, 'label')
print(f"\nBANGLA SPLIT:")
print(f"  Train: {len(bn_train):,} ({len(bn_train)/len(df_bn)*100:.1f}%)")
print(f"  Val:   {len(bn_val):,} ({len(bn_val)/len(df_bn)*100:.1f}%)")
print(f"  Test:  {len(bn_test):,} ({len(bn_test)/len(df_bn)*100:.1f}%)")
print(f"\n  Train label distribution:")
print(f"  {bn_train['label'].value_counts().sort_index().to_dict()}")
print(f"  Val label distribution:")
print(f"  {bn_val['label'].value_counts().sort_index().to_dict()}")
print(f"  Test label distribution:")
print(f"  {bn_test['label'].value_counts().sort_index().to_dict()}")

STRATIFIED SPLITTING (80% train / 10% val / 10% test)

ENGLISH SPLIT:
  Train: 83,455 (80.0%)
  Val:   10,432 (10.0%)
  Test:  10,432 (10.0%)

  Train class distribution:
  {0: 27758, 1: 27883, 2: 27814}
  Val class distribution:
  {0: 3470, 1: 3485, 2: 3477}
  Test class distribution:
  {0: 3470, 1: 3486, 2: 3476}

BANGLA SPLIT:
  Train: 67,009 (80.0%)
  Val:   8,376 (10.0%)
  Test:  8,377 (10.0%)

  Train label distribution:
  {0: 33119, 1: 20477, 2: 13413}
  Val label distribution:
  {0: 4140, 1: 2560, 2: 1676}
  Test label distribution:
  {0: 4140, 1: 2560, 2: 1677}


In [9]:
OUTPUT_DIR = "/kaggle/working/"
 
# Save English splits
en_train[['text_clean', 'class']].to_csv(f"{OUTPUT_DIR}en_train.csv", index=False)
en_val[['text_clean', 'class']].to_csv(f"{OUTPUT_DIR}en_val.csv", index=False)
en_test[['text_clean', 'class']].to_csv(f"{OUTPUT_DIR}en_test.csv", index=False)
 
# Save Bangla splits (include source for analysis)
bn_train[['text_clean', 'label', 'source']].to_csv(f"{OUTPUT_DIR}bn_train.csv", index=False)
bn_val[['text_clean', 'label', 'source']].to_csv(f"{OUTPUT_DIR}bn_val.csv", index=False)
bn_test[['text_clean', 'label', 'source']].to_csv(f"{OUTPUT_DIR}bn_test.csv", index=False)
 
# Also save the full cleaned datasets (needed for Phase 4 annotation selection)
df_en[['text', 'text_clean', 'class']].to_csv(f"{OUTPUT_DIR}en_full_clean.csv", index=False)
df_bn[['text', 'text_clean', 'label', 'source']].to_csv(f"{OUTPUT_DIR}bn_full_clean.csv", index=False)
 
print("All files saved!")
print("\nFiles created:")
print(f"  {OUTPUT_DIR}en_train.csv ({len(en_train):,} rows)")
print(f"  {OUTPUT_DIR}en_val.csv ({len(en_val):,} rows)")
print(f"  {OUTPUT_DIR}en_test.csv ({len(en_test):,} rows)")
print(f"  {OUTPUT_DIR}bn_train.csv ({len(bn_train):,} rows)")
print(f"  {OUTPUT_DIR}bn_val.csv ({len(bn_val):,} rows)")
print(f"  {OUTPUT_DIR}bn_test.csv ({len(bn_test):,} rows)")
print(f"  {OUTPUT_DIR}en_full_clean.csv ({len(df_en):,} rows)")
print(f"  {OUTPUT_DIR}bn_full_clean.csv ({len(df_bn):,} rows)")

All files saved!

Files created:
  /kaggle/working/en_train.csv (83,455 rows)
  /kaggle/working/en_val.csv (10,432 rows)
  /kaggle/working/en_test.csv (10,432 rows)
  /kaggle/working/bn_train.csv (67,009 rows)
  /kaggle/working/bn_val.csv (8,376 rows)
  /kaggle/working/bn_test.csv (8,377 rows)
  /kaggle/working/en_full_clean.csv (104,319 rows)
  /kaggle/working/bn_full_clean.csv (83,762 rows)


In [10]:
print("=" * 70)
print("PHASE 2 COMPLETE!")
print("=" * 70)
print("\nSummary:")
print(f"  English: {len(df_en):,} texts -> {len(en_train):,} train / {len(en_val):,} val / {len(en_test):,} test")
print(f"  Bangla:  {len(df_bn):,} texts -> {len(bn_train):,} train / {len(bn_val):,} val / {len(bn_test):,} test")

PHASE 2 COMPLETE!

Summary:
  English: 104,319 texts -> 83,455 train / 10,432 val / 10,432 test
  Bangla:  83,762 texts -> 67,009 train / 8,376 val / 8,377 test
